In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/Finetune/Embedding/finetune-embedding.ipynb
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/Eval_Ground_truth_GenAnswer.ipynb
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/final_result.json
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/requirements.txt
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/pipeline.py
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/setup.py
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/database/qdrant_manager.py
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/database/__init__.py
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Project/src/config/__init__.py
/kaggle/input/datasets/ngpham06/se365-final3/SE365_Final1/SE365/Repo Proje

In [2]:
version_rerank = "FN"

In [3]:
import sys
import os
import pandas as pd
import sys
import os
import gc
import torch
import json
from tqdm import tqdm
module_path = "/kaggle/input/datasets/ngpham06/se365-final2-genanswer/SE365_Final1/SE365/Repo Project/src"
os.chdir(module_path)
sys.path.append(module_path)
!ls

config				   final_result.json  rerank
data				   fusion	      retrieval
database			   generation	      rewrite
Eval_Ground_truth_GenAnswer.ipynb  pipeline.py	      setup.py
extract				   requirements.txt


In [4]:
!pip install -qr requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.4/483.4 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 

# Dataset

In [5]:
from pipeline import qa_pipeline

2026-06-22 09:00:49.008221: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782118849.191344      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782118849.243281      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782118849.674754      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782118849.674803      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782118849.674806      23 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
df_test = pd.read_csv('/kaggle/input/datasets/ngpham06/groundtruth-genanswers/ground_truth_genAnswers.csv')

# HS + RW + 1 RRv3 (Best exp in Retrieval)

In [ ]:
# Tên
version = "GenAns"

# Chạy
final_result = []

for d in tqdm(df_test['question']):
  result = qa_pipeline(
    user_query=d,
      
    # rewrite
    use_rewrite = True,
    num_rewrites = 3,
    # retrieval
    retrieval_top_k = 100,  # -> 400
    # rewrite rrf
    use_rewrite_rrf = True,  # -> 50
    rewrite_rrf_top_n = 50,

    # rerank
    use_rerank = True, # -> 10
    # dual rerank
    use_dual_rerank = False, # mỗi rerank -> 50
    rerank_top_k = 10, # -> fusion 2 rerank lại -> 10
      
    # extract
    use_extract = False,
    # print result (in thông số khi pipeline thực hiện từng bước)
    print_result = False
)
  final_result.append(result)

# Lưu
output_filename = f'/kaggle/working/final_result_{version}_{version_rerank}.json'

with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(final_result, f, ensure_ascii=False, indent=4)

print(f"'final_result' has been saved to {output_filename}")

  0%|          | 0/564 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[REWRITE] 4 queries
[HYBRID SEARCH] 4 done
[REWRITE RRF] 50 docs
[RERANK] single done 10


  0%|          | 1/564 [00:15<2:21:46, 15.11s/it]

[LLM] done
[REWRITE] 4 queries
[HYBRID SEARCH] 4 done
[REWRITE RRF] 50 docs
[RERANK] single done 10


  0%|          | 2/564 [00:26<2:01:49, 13.01s/it]

...

[LLM] done
[REWRITE] 4 queries
[HYBRID SEARCH] 4 done
[REWRITE RRF] 50 docs
[RERANK] single done 10


100%|█████████▉| 563/564 [2:06:21<00:17, 17.36s/it]

[LLM] done
[REWRITE] 4 queries
[HYBRID SEARCH] 4 done
[REWRITE RRF] 50 docs
[RERANK] single done 10


100%|██████████| 564/564 [2:06:40<00:00, 13.48s/it]

[LLM] done
'final_result' has been saved to /kaggle/working/final_result_Full_1RR_GenAns_FN.json
